In [2]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\zahee\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [8]:
data = pd.read_csv('dataset.csv', encoding='latin-1')

In [9]:
data.head()

,email,category
0,"URL: http://www.newsisfree.com/click/-1,817167...",not-spam
1,"On Thu, 19 Sep 2002, Bill Stoddard wrote:\n\n-...",not-spam
2,Dan Kohn <dan@dankohn.com> writes:\n\n\n\n> Gu...,not-spam
3,wintermute wrote:\n\n>>Anyone know where in Ir...,not-spam
4,"I attended the same conference, and was impres...",not-spam


In [10]:
data.shape

(3796, 2)

In [11]:
data['category'].value_counts()

category
not-spam    1900
spam        1896
Name: count, dtype: int64

In [12]:
data['label'] = data['category'].map({ 'not-spam':0, 'spam':1})

In [13]:
data.head()

,email,category,label
0,"URL: http://www.newsisfree.com/click/-1,817167...",not-spam,0
1,"On Thu, 19 Sep 2002, Bill Stoddard wrote:\n\n-...",not-spam,0
2,Dan Kohn <dan@dankohn.com> writes:\n\n\n\n> Gu...,not-spam,0
3,wintermute wrote:\n\n>>Anyone know where in Ir...,not-spam,0
4,"I attended the same conference, and was impres...",not-spam,0


In [18]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(text):
    text = text.lower()

    # remove punctuation and numbers
    text = re.sub(r'[^a-zA-Z\s]','',text)

    # tokenize
    words = text.split()

    # remove stop words and perform stemming
    words = [
        stemmer.stem(word) for word in words if word not in stop_words
    ]

    return ' '.join(words)

    

In [20]:
data['clean_message'] = data['email'].apply(clean_text)

In [21]:
data.head()

,email,category,label,clean_message
0,"URL: http://www.newsisfree.com/click/-1,817167...",not-spam,0,url httpwwwnewsisfreecomclick date suppli deta...
1,"On Thu, 19 Sep 2002, Bill Stoddard wrote:\n\n-...",not-spam,0,thu sep bill stoddard wrote like chang someon ...
2,Dan Kohn <dan@dankohn.com> writes:\n\n\n\n> Gu...,not-spam,0,dan kohn dandankohncom write guy habea infring...
3,wintermute wrote:\n\n>>Anyone know where in Ir...,not-spam,0,wintermut wrote anyon know ireland get replac ...
4,"I attended the same conference, and was impres...",not-spam,0,attend confer impress system jim didnt mention...


In [22]:
x = data['clean_message']
y = data['label']

In [23]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [24]:
vectorizer = TfidfVectorizer()
x_train_vector = vectorizer.fit_transform(x_train)
x_test_vector = vectorizer.transform(x_test)


In [26]:
model = MultinomialNB()
model.fit(x_train_vector, y_train)


,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [27]:
y_predict = model.predict(x_test_vector)

In [28]:
accuracy = accuracy_score(y_test, y_predict)
precision = precision_score(y_test, y_predict)
recall = recall_score(y_test, y_predict)
f1 = f1_score(y_test, y_predict)

In [29]:
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

Accuracy : 0.9828947368421053
Precision: 0.9919571045576407
Recall   : 0.9736842105263158
F1 Score : 0.9827357237715804


In [31]:
print(classification_report(
    y_test,
    y_predict,
    target_names=['Not Spam', 'Spam']
))

              precision    recall  f1-score   support

    Not Spam       0.97      0.99      0.98       380
        Spam       0.99      0.97      0.98       380

    accuracy                           0.98       760
   macro avg       0.98      0.98      0.98       760
weighted avg       0.98      0.98      0.98       760



In [37]:
import pickle

with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)